# Routing — 01: PG primary · ES INDEX · Parquet BACKUP

**Persona:** Platform engineer wiring a multi-sink ingestion pipeline.


**Goal:** Demonstrate the role-based driver refactor's routing composition:
- PostgreSQL as the **Primary** for WRITE and READ (source of truth).
- Elasticsearch as an **INDEX** target — async search/mirror, fed the raw envelope.
- DuckDB (or the file-sink of choice) as a **BACKUP** target with `fmt=parquet`.

All three live under a single `CollectionRoutingConfig`.  INDEX and BACKUP
are declared under `metadata.operations` so the ReindexWorker / backup endpoints
pick them up without blocking the synchronous WRITE path.

## Architecture

```
           POST /items
               │
        ┌──────▼──────┐
        │   Router    │
        └──────┬──────┘
               │ WRITE (sync)
        ┌──────▼──────┐
        │  PG primary │ ───────► source of truth (READ also here)
        └──────┬──────┘
               │ post-commit notify
       ┌───────┴───────┐
       ▼               ▼
  INDEX (ES)     BACKUP (Parquet)
   async           async
```


In [ ]:
import os
import json
import uuid as _uuid
import time as _t

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL    = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)

headers = {"Authorization": f"Bearer {ADMIN_TOKEN}"} if ADMIN_TOKEN else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

CATALOG_ID    = f"rtng01-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID = "events"
print(f"Using catalog={CATALOG_ID} collection={COLLECTION_ID}")


In [ ]:
# Bootstrap: create the ephemeral catalog+collection.

catalog = {"id": CATALOG_ID, "type": "Catalog", "title": "Routing demo", "description": "PG+ES+Parquet fan-out example", "stac_version": "1.1.0", "conformsTo": [], "links": []}
for _ in range(3):
    r = client.post("/stac/catalogs", content=json.dumps(catalog))
    if r.status_code in (200, 201, 409):
        break
assert r.status_code in (200, 201, 409), r.text[:400]

collection = {
    "id": COLLECTION_ID, "type": "Collection",
    "title": "Event stream", "description": "Ingest events fanned out to ES + Parquet",
    "stac_version": "1.1.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [[None, None]]}},
    "license": "proprietary", "keywords": ["events"], "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection),
)
assert r.status_code in (200, 201, 409), r.text[:400]
print("Catalog + collection ready")


## Step 1 — Pin the PostgreSQL driver as the primary

Default `ItemsPostgresqlDriverConfig` — attribute + geometry sidecars are
injected automatically via the registry (no explicit `sidecars` needed for
default VECTOR collections).

In [ ]:
pg_cfg = {
    "class_key": "ItemsPostgresqlDriverConfig",
    # default VECTOR collection → geometry + attributes sidecars auto-injected
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsPostgresqlDriverConfig",
    content=json.dumps(pg_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]


## Step 2 — Wire ES (obfuscated) as an INDEX target

The Elasticsearch obfuscated driver (`ItemsElasticsearchObfuscatedDriver`) is a
search-tier mirror.  Registering it under `metadata.operations.INDEX` makes the
ReindexWorker fan out on post-commit; `transformed=false` means the raw envelope
(PG's primary shape) is forwarded — no TRANSFORM chain is applied.

In [ ]:
es_cfg = {
    "class_key": "ItemsElasticsearchObfuscatedDriverConfig",
    "target": {"index_prefix": "rtng-"},
}
# Many deployments accept the PUT even when the driver module isn't loaded; the
# _validate_routing_entries apply-handler catches that, with a clearer 400 than
# the raw "module missing" error.
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsElasticsearchObfuscatedDriverConfig",
    content=json.dumps(es_cfg),
)
print(f"ES config PUT → {r.status_code}")


## Step 3 — Wire DuckDB as a BACKUP target emitting Parquet

DuckDB's BACKUP-role behaviour uses its `EXPORT` capability.  The routing entry
declares `fmt="parquet"` so ingestion events carry the sink into the backup
endpoint (`GET .../backup?format=parquet`).

In [ ]:
duck_cfg = {
    "class_key": "ItemsDuckdbDriverConfig",
    "warehouse": "/tmp/duckdb-backups",
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/ItemsDuckdbDriverConfig",
    content=json.dumps(duck_cfg),
)
print(f"DuckDB config PUT → {r.status_code}")


## Step 4 — Build the composite routing config

One `CollectionRoutingConfig` ties it all together.  `operations` covers the
top-level WRITE/READ path; `metadata.operations` carries the async INDEX / BACKUP
entries.  Fan-out is async-by-role: the ReindexWorker consumes INDEX entries and
the backup endpoint consumes BACKUP entries.


In [ ]:
routing_cfg = {
    "class_key": "CollectionRoutingConfig",
    "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ":  [{"driver_id": "ItemsPostgresqlDriver"}],
    },
    "metadata": {
        "operations": {
            # INDEX — search mirror; transformed=False means "feed the raw envelope"
            "INDEX": [{
                "driver_id": "ItemsElasticsearchObfuscatedDriver",
                "transformed": False,
                "on_failure": "warn",
            }],
            # BACKUP — file-sink; fmt binds the sink to the ?format=parquet query.
            "BACKUP": [{
                "driver_id": "ItemsDuckdbDriver",
                "transformed": False,
                "fmt": "parquet",
                "on_failure": "warn",
            }],
        },
    },
}
r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    content=json.dumps(routing_cfg),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print("Routing configured: PG primary · ES INDEX · DuckDB BACKUP(fmt=parquet)")


## Step 5 — Verify effective routing (read the resolved config)

The config waterfall merges platform → catalog → collection — verifying the
collection scope resolves correctly is a sanity check before pushing data.

In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig/effective"
)
assert r.status_code == 200, r.text[:400]
effective = r.json()
print(json.dumps(effective, indent=2)[:1200])


## Step 6 — Write a feature; observe WRITE returns immediately after PG commits

INDEX and BACKUP are async — the user request returns on PG success and the
other sinks catch up on the ReindexWorker / backup endpoint read, respectively.


In [ ]:
feat = {
    "type": "Feature",
    "id": f"evt-{_uuid.uuid4().hex[:8]}",
    "geometry": {"type": "Point", "coordinates": [12.4924, 41.8902]},  # Rome
    "properties": {"event_type": "notebook_demo", "severity": "info"},
}
r = client.post(
    f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    content=json.dumps({"type": "FeatureCollection", "features": [feat]}),
)
assert r.status_code in (200, 201, 204), r.text[:400]
print(f"WRITE → {r.status_code}; PG commit done (ES/DuckDB propagation is async)")


## Teardown

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}")
print(f"DELETE catalog → {r.status_code}")
client.close()
